In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder

In [4]:
iris = load_iris()
X, y = iris.data, iris.target.reshape(-1,1)
scaler = StandardScaler()
X_norm = scaler.fit_transform(X)
encoder = OneHotEncoder(sparse_output=False)
y_cat = encoder.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X_norm, y_cat, test_size=0.2, random_state=42)

In [5]:
tf_model = models.Sequential([
    layers.Input(shape=(4,)),
    layers.Dense(8, activation='relu'),
    layers.Dense(3, activation='softmax')
])
tf_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
tf_model.fit(X_train, y_train, epochs=50, verbose=0)
loss, acc = tf_model.evaluate(X_test, y_test, verbose=0)
print(f"TensorFlow Test Accuracy: {acc:.4f}")

TensorFlow Test Accuracy: 0.9333


In [6]:
le = LabelEncoder()
y_idx = le.fit_transform(iris.target)
X_train_pt, X_test_pt, y_train_pt, y_test_pt = train_test_split(
    torch.tensor(X_norm, dtype=torch.float32),
    torch.tensor(y_idx, dtype=torch.long),
    test_size=0.2,
    random_state=42
)

In [7]:
class IrisNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(8, 3)
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

In [8]:
pt_model = IrisNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(pt_model.parameters(), lr=0.01)

In [9]:
for epoch in range(100):
    optimizer.zero_grad()
    outputs = pt_model(X_train_pt)
    loss = criterion(outputs, y_train_pt)
    loss.backward()
    optimizer.step()

In [10]:
with torch.no_grad():
    preds = pt_model(X_test_pt).argmax(dim=1)
    acc_pt = (preds == y_test_pt).float().mean().item()
print(f"PyTorch Test Accuracy: {acc_pt:.4f}")

PyTorch Test Accuracy: 1.0000
